In [ ]:
import requests
import json
from pathlib import Path
from datetime import datetime, timedelta

# Resolve paths from either the project root or the scripts folder.
PROJECT_ROOT = next(
    (folder for folder in (Path.cwd(), *Path.cwd().parents)
     if (folder / "json_data_file").is_dir() and (folder / "scripts").is_dir()),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Open this notebook from the Nordic Energy project folder.")
RAW_DATA_PATH = PROJECT_ROOT / "json_data_file" / "prices.json"


In [ ]:


base_url = "https://api.energidataservice.dk/dataset/DayAheadPrices"

def get_day_ahead_prices(start_date, end_date):

    # 1. Define what data we want
    params = {
        "start": start_date,
        "end": end_date,
        "filter": json.dumps({
            "PriceArea": ["DK1", "DK2"]
        }),
        "sort": "TimeUTC asc",
        "limit": 0
    }

    # 2. Send the HTTP request
    response = requests.get(
        base_url,
        params=params,
        timeout=30
    )

    # 3. Check for HTTP errors
    response.raise_for_status()

    # 4. Convert JSON response to Python objects
    data = response.json()

    # 5. Extract the records
    records = data.get("records", [])

    # 6. Show what happened
    print("API request successful!")
    
    # 7. Return data to the caller
    return data


# Reuse the saved snapshot so Run All preserves existing data.
if __name__ == "__main__":
    if RAW_DATA_PATH.exists():
        with RAW_DATA_PATH.open("r", encoding="utf-8") as file:
            data = json.load(file)
        print("Loaded existing JSON snapshot.")
    else:
        data = get_day_ahead_prices("2026-11-01", "2026-11-08")
        # Exclusive creation protects against accidentally replacing a snapshot.
        with RAW_DATA_PATH.open("x", encoding="utf-8") as file:
            json.dump(data, file, indent=2)
        print("JSON saved successfully!")

    print("Records:", len(data["records"]))


In [35]:

# data quality check function
def check_missing_prices(file_path):

    with open(file_path, "r") as file:
        data = json.load(file)

    records = data.get("records", [])

    columns = [
        "TimeUTC",
        "PriceArea",
        "DayAheadPriceEUR",
        "DayAheadPriceDKK"
    ]

    results = {}

    for column in columns:
        missing_count = sum(
            1 for record in records
            if record.get(column) is None
        )

        results[column] = {
            "missing_count": missing_count,
            "missing_percentage": (
                missing_count / len(records) * 100
                if records else 0
            )
        }

    return results


In [ ]:
data_quality = check_missing_prices(
    RAW_DATA_PATH
)

print(data_quality)

In [ ]:
# check duplicate records function 
# Duplicate Key Detection using a composite key
# (TimeUTC + PriceArea)
def check_duplicates(records):

    seen = set()
    duplicate_count = 0

    for record in records:
        key = (record["TimeUTC"], record["PriceArea"])

        if key in seen:
            duplicate_count += 1
        else:
            seen.add(key)

    return {
        "total_records": len(records),
        "duplicates": duplicate_count
    }


In [42]:
result = check_duplicates(data["records"])
print(result)

{'total_records': 1344, 'duplicates': 0}


In [43]:
# Data quality check - validate price areas

def check_price_areas(records):

    # Define the price areas we want to accept
    valid_areas = ["DK1", "DK2"]

    # Start counting invalid records from zero
    invalid_count = 0

    # Go through each record and check the PriceArea
    for record in records:

        # If PriceArea is not DK1 or DK2, count it as invalid
        if record["PriceArea"] not in valid_areas:
            invalid_count += 1

    # Return the results of the validation
    return {
        "total_records": len(records),
        "invalid_areas": invalid_count
    }

In [44]:
result = check_price_areas(data["records"])
print(result)

{'total_records': 1344, 'invalid_areas': 0}


In [45]:

# Data quality check - validate price data types

def check_price_types(records):

    # Define the columns that should contain numbers
    price_columns = [
        "DayAheadPriceEUR",
        "DayAheadPriceDKK"
    ]

    # Store the results for each column
    results = {}

    # Check each price column
    for column in price_columns:

        invalid_count = 0

        # Go through all records
        for record in records:

            # Check if the value is not an integer or float
            if not isinstance(record.get(column), (int, float)):
                invalid_count += 1

        # Save the result for this column
        results[column] = invalid_count

    return results


In [46]:
result = check_price_types(data["records"])
print(result)

{'DayAheadPriceEUR': 0, 'DayAheadPriceDKK': 0}


In [49]:

# Data quality check - validate timestamps

def check_timestamps(records):

    # Start counting invalid timestamps
    invalid_count = 0

    # Go through each record
    for record in records:

        # Get the timestamp value
        timestamp = record.get("TimeUTC")

        try:
            # Check if the timestamp has a valid date and time
            datetime.strptime(timestamp, "%Y-%m-%dT%H:%M:%S")

        except (ValueError, TypeError):
            # Count the timestamp if it is invalid
            invalid_count += 1

    # Return the validation results
    return {
        "total_records": len(records),
        "invalid_timestamps": invalid_count
    }


In [50]:
result = check_timestamps(data["records"])
print(result)

{'total_records': 1344, 'invalid_timestamps': 0}


In [55]:


# Data quality check - find missing 15-minute intervals

def check_time_intervals(records, start_date, end_date):

    # Store the existing timestamp and price area combinations
    existing_keys = set()

    for record in records:
        key = (record["TimeDK"], record["PriceArea"])
        existing_keys.add(key)

    # Define the start and end dates
    current = datetime.strptime(start_date, "%Y-%m-%d")
    end = datetime.strptime(end_date, "%Y-%m-%d")

    # Start counting expected and missing records
    expected_count = 0
    missing_count = 0

    # Check every 15-minute interval until the end date
    while current < end:

        timestamp = current.strftime("%Y-%m-%dT%H:%M:%S")

        # We expect one record for each price area
        for area in ["DK1", "DK2"]:

            expected_count += 1

            # Count the record if the timestamp and area are missing
            if (timestamp, area) not in existing_keys:
                missing_count += 1

        # Move to the next 15-minute interval
        current += timedelta(minutes=15)

    return {
        "expected_records": expected_count,
        "missing_intervals": missing_count
    }


In [56]:
result = check_time_intervals(
    data["records"],
    "2026-11-01",
    "2026-11-08"
)

print(result)

{'expected_records': 1344, 'missing_intervals': 0}
